# 🖊️ Inference Local — Handwriting Agent Evaluation

| Section | Purpose |
|---|---|
| **1 — Setup** | Imports, config, LLM client |
| **2 — Run Episodes** | Execute easy / medium / hard, collect per-step metrics |
| **3 — Visualisation** | Coverage graph, pixel efficiency, summary |


## Section 1 — ⚙️ Setup & Configuration


In [2]:
import os, json, asyncio, textwrap
import sys; sys.path.insert(0, os.path.abspath('..'))
from typing import List, Optional, Literal

import nest_asyncio
nest_asyncio.apply()

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()
from learn_handwriting import LearnHandwritingAction, LearnHandwritingEnv

print('✅ Imports OK')


/Users/abhijeetmishra/PycharmProjects/hackathon/scaler_8_april/learn_handwriting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports OK


In [3]:
API_KEY      = os.getenv('HF_TOKEN') or os.getenv('API_KEY')
API_BASE_URL = os.getenv('API_BASE_URL', 'https://router.huggingface.co/v1')
MODEL_NAME   = os.getenv('MODEL_NAME')
ENV_BASE_URL = os.getenv('ENV_BASE_URL', 'http://localhost:8000')
MAX_STEPS    = 15
TEMPERATURE  = 0.7
ALL_TASKS    = ['easy', 'medium', 'hard']
MAX_DRAWN_MULTIPLIER = float(os.getenv('MAX_DRAWN_MULTIPLIER', '1.7'))

print(f'Model  : {MODEL_NAME}')
print(f'Env URL: {ENV_BASE_URL}')
print(f'Tasks  : {ALL_TASKS}')


Model  : Qwen/Qwen2.5-72B-Instruct
Env URL: http://localhost:8000
Tasks  : ['easy', 'medium', 'hard']


In [4]:
class StrokeOutput(BaseModel):
    reasoning:   str = Field(default='')
    action_type: Literal['line', 'curve', 'circle', 'ellipse'] = 'line'
    x1: int = 10; 
    y1: int = 10
    x2: Optional[int] = None; 
    y2: Optional[int] = None
    x3: Optional[int] = None; 
    y3: Optional[int] = None
    radius: Optional[int] = None
    rx: Optional[int] = None; 
    ry: Optional[int] = None


SYSTEM_PROMPT = textwrap.dedent(f"""
    You are drawing capital letters on a 100x100 pixel canvas.
    Coordinate system: x=0 left, x=99 right, y=0 top, y=99 bottom.

    Actions:
    - 'line'   : straight line (x1,y1) -> (x2,y2)
    - 'curve'  : bezier (x1,y1) -> (x2,y2) passing through (x3,y3)
    - 'circle' : circle centred (x1,y1) with `radius`
    - 'ellipse': oval centred (x1,y1) with `rx` (horizontal) and `ry` (vertical)

    Rules:
    - Max {MAX_STEPS} actions. Goal: cover 90% of the target character pixels.
    - INK PENALTY: fail if you draw more than {MAX_DRAWN_MULTIPLIER}x the target pixels.
    - CRITICAL for circle/ellipse: centre +/- radius must stay within [0,99] on BOTH axes.
      Safe example: ellipse centred (50,50) rx=30 ry=35

    SHAPE INTEGRITY (do NOT fill these regions):
    A=inner triangle, B=two lobe holes, O/Q=circle interior,
    C/G=right-side opening, S=two bridge gaps.
    Covering >60% of a protected region ends the episode with reward=0.

    Respond ONLY with a valid JSON object, no markdown.
""").strip()


def _action_str(s: StrokeOutput) -> str:
    if s.action_type == 'circle':
        return f'circle({s.x1},{s.y1},r={s.radius})'
    if s.action_type == 'ellipse':
        return f'ellipse({s.x1},{s.y1},rx={s.rx},ry={s.ry})'
    if s.action_type == 'curve':
        return f'curve({s.x1},{s.y1}->{s.x2},{s.y2})'
    return f'line({s.x1},{s.y1}->{s.x2},{s.y2})'


def get_stroke(client, step, target_char, match_pct,
               last_matched, last_wasted, ink_remaining,
               last_reward, history, integrity_violated=False) -> StrokeOutput:
    integrity_warn = '\n⚠️ LAST ACTION VIOLATED INTEGRITY!' if integrity_violated else ''
    user_msg = textwrap.dedent(f"""
        Draw capital letter: {target_char}
        Step {step}/{MAX_STEPS}  |  Coverage: {match_pct:.1%}  (goal 90%)
        Last action: matched={last_matched}px  wasted={last_wasted}px  ink_left={ink_remaining}{integrity_warn}
        History (last 5):\n{chr(10).join(history[-5:]) or 'None yet'}
    """).strip()
    try:
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': user_msg},
            ],
            temperature=TEMPERATURE,
            response_format={'type': 'json_object'},
        )
        d = json.loads(resp.choices[0].message.content or '{}')
        s = StrokeOutput(
            reasoning   = d.get('reasoning', ''),
            action_type = d.get('action_type', 'line'),
            x1=int(d.get('x1', 10)), y1=int(d.get('y1', 10)),
            x2=int(d['x2']) if d.get('x2') is not None else None,
            y2=int(d['y2']) if d.get('y2') is not None else None,
            x3=int(d['x3']) if d.get('x3') is not None else None,
            y3=int(d['y3']) if d.get('y3') is not None else None,
            radius=int(d['radius']) if d.get('radius') is not None else None,
            rx=int(d['rx']) if d.get('rx') is not None else None,
            ry=int(d['ry']) if d.get('ry') is not None else None,
        )
        # Defensive clamp
        s.x1 = max(0, min(99, s.x1)); s.y1 = max(0, min(99, s.y1))
        for attr in ('x2','y2','x3','y3'):
            v = getattr(s, attr)
            if v is not None: setattr(s, attr, max(0, min(99, v)))
        if s.radius is not None: s.radius = max(1, min(100, s.radius))
        if s.rx     is not None: s.rx     = max(1, min(100, s.rx))
        if s.ry     is not None: s.ry     = max(1, min(100, s.ry))
        return s
    except Exception as exc:
        print(f'  [WARN] LLM error: {exc}')
        return StrokeOutput(reasoning='fallback', action_type='line',
                            x1=10, y1=10, x2=90, y2=90)


client = OpenAI(base_url=API_BASE_URL, api_key=API_KEY)
print('✅ LLM client ready')


✅ LLM client ready


## Section 2 — 🏃 Run Episodes


In [5]:
async def run_episode(task_name: str, env, client) -> dict:
    """Run one episode and return detailed per-step metrics."""
    result = await env.reset(task=task_name)
    obs    = result.observation
    char   = obs.target_character

    episode = {
        'task': task_name, 'character': char,
        'steps': [], 'final_score': 0.0, 'success': False,
    }

    history, last_matched, last_wasted = [], 0, 0
    ink_remaining, last_reward, last_integrity = getattr(obs,'ink_remaining',9999), 0.0, False

    print(f"\n{'─'*60}")
    print(f"  Task: {task_name.upper():<8}  Character: '{char}'")
    print(f"{'─'*60}")
    print(f"  {'Step':<5} {'Action':<38} {'Coverage':>9} {'Match':>7} {'Waste':>7} {'Reward':>8}")
    print(f"  {'─'*4} {'─'*37} {'─'*9} {'─'*7} {'─'*7} {'─'*8}")

    for step in range(1, MAX_STEPS + 1):
        if result.done:
            break

        stroke = get_stroke(
            client, step, char, obs.match_percentage,
            last_matched, last_wasted, ink_remaining,
            last_reward, history, last_integrity,
        )

        action = LearnHandwritingAction(
            action_type=stroke.action_type,
            x1=stroke.x1, y1=stroke.y1,
            x2=stroke.x2, y2=stroke.y2,
            x3=stroke.x3, y3=stroke.y3,
            radius=stroke.radius, rx=stroke.rx, ry=stroke.ry,
        )
        astr = _action_str(stroke)

        result = await env.step(action)
        obs    = result.observation
        reward = result.reward or 0.0

        last_matched  = obs.pixels_matched_this_stroke
        last_wasted   = getattr(obs, 'pixels_wasted_this_stroke', 0)
        ink_remaining = getattr(obs, 'ink_remaining', 0)
        last_integrity = getattr(obs, 'integrity_violated', False)
        last_reward    = reward
        cov            = obs.match_percentage * 100

        flag = '  ⚠️ INTEGRITY' if last_integrity else ''
        print(f"  {step:<5} {astr:<38} {cov:>8.1f}% {last_matched:>7} {last_wasted:>7} {reward:>8.4f}{flag}")
        print(f"        💭 {stroke.reasoning[:90]}")

        history.append(
            f"Step {step}: {astr} → matched={last_matched}px "
            f"coverage={cov:.1f}% reward={reward:.4f}"
        )
        episode['steps'].append({
            'step': step, 'action': astr, 'reasoning': stroke.reasoning,
            'reward': reward, 'coverage_pct': cov,
            'matched_this_step': last_matched,
            'wasted_this_step':  last_wasted,
            'ink_remaining':     ink_remaining,
            'integrity_violated': last_integrity,
        })

    score = min(max(obs.match_percentage, 0.001), 0.999)
    episode['final_score'] = score
    episode['success']     = score >= 0.90

    status = '✅ SUCCESS' if episode['success'] else '❌ FAILED'
    print(f"\n  {status}  —  Final coverage: {score:.1%}  Steps: {len(episode['steps'])}/{MAX_STEPS}")
    return episode

print('✅ run_episode() defined')


✅ run_episode() defined


In [6]:
async def run_all_tasks() -> List[dict]:
    episodes = []
    for task in ALL_TASKS:
        env = LearnHandwritingEnv(base_url=ENV_BASE_URL)
        try:
            ep = await run_episode(task, env, client)
            episodes.append(ep)
        finally:
            try:
                await env.close()
            except Exception:
                pass
    return episodes


# ── RUN ──────────────────────────────────────────────────────────────────────
# Make sure your environment server is running: `openenv serve` or docker
results: List[dict] = asyncio.run(run_all_tasks())
print(f'\n✅ All {len(results)} episodes complete.')



────────────────────────────────────────────────────────────
  Task: EASY      Character: 'T'
────────────────────────────────────────────────────────────
  Step  Action                                  Coverage   Match   Waste   Reward
  ──── ───────────────────────────────────── ───────── ─────── ─────── ────────


AttributeError: 'LearnHandwritingAction' object has no attribute 'width'

## Section 3 — 📊 Analysis & Visualisation


In [ ]:
# ── Plot 1: Coverage % Progression ───────────────────────────────────────────
TASK_COLORS = {'easy': '#4caf50', 'medium': '#ff9800', 'hard': '#f44336'}

fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 5), sharey=True)
if len(results) == 1:
    axes = [axes]

for ax, ep in zip(axes, results):
    steps   = [s['step']        for s in ep['steps']]
    cov     = [s['coverage_pct'] for s in ep['steps']]
    integ   = [s['step'] for s in ep['steps'] if s['integrity_violated']]

    color = TASK_COLORS.get(ep['task'], '#888')
    ax.plot(steps, cov, marker='o', color=color, linewidth=2.5, label='Coverage %')
    ax.fill_between(steps, cov, alpha=0.15, color=color)
    ax.axhline(90, color='crimson', linestyle='--', linewidth=1.5, label='90% goal')

    for bad_step in integ:
        ax.axvline(bad_step, color='crimson', alpha=0.6, linewidth=1, linestyle=':')
        ax.annotate('⚠️', xy=(bad_step, 5), ha='center', fontsize=9, color='crimson')

    score_pct = ep['final_score'] * 100
    status    = '✅' if ep['success'] else '❌'
    ax.set_title(
        f"{status} {ep['task'].upper()} — '{ep['character']}'\n"
        f"Final: {score_pct:.1f}%  Steps: {len(ep['steps'])}/{MAX_STEPS}",
        fontsize=11, fontweight='bold',
    )
    ax.set_xlabel('Step', fontsize=10)
    ax.set_ylabel('Coverage (%)', fontsize=10)
    ax.set_xlim(0.5, MAX_STEPS + 0.5)
    ax.set_ylim(0, 105)
    ax.set_xticks(range(1, MAX_STEPS + 1))
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle('Coverage Progression per Task', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('coverage_progression.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: coverage_progression.png')


In [ ]:
# ── Plot 2: Matched vs Wasted Pixels per Step ─────────────────────────────────
fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 5), sharey=False)
if len(results) == 1:
    axes = [axes]

for ax, ep in zip(axes, results):
    steps   = [s['step']              for s in ep['steps']]
    matched = [s['matched_this_step'] for s in ep['steps']]
    wasted  = [s['wasted_this_step']  for s in ep['steps']]
    x = np.arange(len(steps))
    w = 0.35

    ax.bar(x - w/2, matched, w, label='Matched (on-target)', color='#4caf50', alpha=0.85)
    ax.bar(x + w/2, wasted,  w, label='Wasted (off-target)', color='#f44336', alpha=0.85)

    total_matched = sum(matched)
    total_wasted  = sum(wasted)
    efficiency    = total_matched / max(total_matched + total_wasted, 1) * 100

    ax.set_title(
        f"{ep['task'].upper()} — '{ep['character']}'\n"
        f"Ink efficiency: {efficiency:.1f}%  "
        f"(matched={total_matched}, wasted={total_wasted})",
        fontsize=11,
    )
    ax.set_xlabel('Step', fontsize=10)
    ax.set_ylabel('Pixels', fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels([str(s) for s in steps])
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle('Pixels Matched vs Wasted per Step', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('pixel_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: pixel_efficiency.png')


In [ ]:
# ── Plot 3: Summary + Reasoning Transcript ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

tasks  = [ep['task']         for ep in results]
chars  = [ep['character']    for ep in results]
scores = [ep['final_score'] * 100 for ep in results]
colors = [
    '#4caf50' if s >= 90 else '#ff9800' if s >= 50 else '#f44336'
    for s in scores
]
labels = [f"{t.upper()} — '{c}'" for t, c in zip(tasks, chars)]

bars = ax.barh(labels, scores, color=colors, height=0.5, edgecolor='white')
ax.axvline(90, color='crimson', linestyle='--', linewidth=1.5, label='90% success')
ax.set_xlim(0, 110)
ax.set_xlabel('Final Coverage (%)', fontsize=11)
ax.set_title('Episode Summary', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)

for bar, score in zip(bars, scores):
    status = '✅' if score >= 90 else '❌'
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}% {status}', va='center', fontsize=10)

legend_patches = [
    mpatches.Patch(color='#4caf50', label='≥ 90% — Success'),
    mpatches.Patch(color='#ff9800', label='50–89% — Partial'),
    mpatches.Patch(color='#f44336', label='< 50% — Failed'),
]
ax.legend(handles=legend_patches, fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('episode_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: episode_summary.png')

# ── Reasoning Transcript ──────────────────────────────────────────────────────
print('\n' + '='*65)
print('  REASONING TRANSCRIPT')
print('='*65)
for ep in results:
    print(f"\n  [{ep['task'].upper()}] '{ep['character']}' — {ep['final_score']:.1%}")
    for s in ep['steps']:
        flag = '⚠️ ' if s['integrity_violated'] else '  '
        print(f"  {flag}Step {s['step']:>2}: {s['action']:<38} "
              f"cov={s['coverage_pct']:>5.1f}%  reward={s['reward']:.4f}")
        print(f"         💭 {s['reasoning'][:95]}")
